<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #3 — Click Capture by Position Tier.** *"These are portfolio-level weighted CTRs
computed from total clicks divided by total impressions in each position tier"* — Top 3 0.423%,
Page 1 0.339%, Striking Distance 0.325%, Page 3-5 0.163%, Deep 0.050%, an 88% drop top-to-bottom.

**My methodology question:** the paper already discloses *how* the statistic is built (pooled
portfolio-wide, not per-brand-then-averaged) — that's good, disclosed practice. The open question
is *concentration*: across 57 brands, how much of each tier's clicks and impressions come from
just one or two high-volume brands? A pooled "total clicks ÷ total impressions" ratio can be
quietly dominated by whichever brand has the most traffic in that tier, so the reported 0.423%
"Top 3" benchmark could describe one large brand far better than it describes a typical smaller
one. This is the exact question I had to answer for my own `expected_ctr_for_tier` in
`w04_baseline_score.ipynb` — I computed it pooled across clients too (same choice, for the same
practical reasons), but I'd want to see a per-brand spread or at least a max-brand-share number
before treating 0.423% as *the* Top 3 benchmark rather than *a* portfolio-weighted one.

**Finding #10 — AI Model Performance (OpenAI vs. Gemini).** The paper explicitly age-controls this
comparison (*"Content age confounds model-performance comparisons"* is named in the Confounding
Variables section) and finds neither provider wins universally once age is controlled — genuinely
careful practice.

**My methodology question:** the Confounding Variables section names content age but not client
(brand) as a possible confound, and the two provider cohorts are very different sizes (OpenAI
145.5K pages, Gemini 91.4K). If a handful of high-volume brands standardized on one provider
company-wide, the "OpenAI vs. Gemini" comparison could partly be a "which brands use which tool"
comparison wearing a provider-comparison costume — the same confound a naive, ungrouped
train/test split would introduce into a model, just showing up in a portfolio comparison instead.
Section 2 below is exactly this check, run on my own Week-5 model: a naive split moved my AUC and
precision@K numbers even with only 27 clients total, purely from letting a few clients' pages
leak across the train/test boundary. I'd want to see the provider comparison re-run within-brand
(or at least see the brand concentration per provider) before fully trusting it stands alone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Grouped, not time-aware**: a documented choice. Same pipeline, same
hyperparameters, same 23 features as `w05_model.ipynb` (`LogisticRegression`, and a
`DecisionTreeClassifier(max_depth=3, min_samples_leaf=200)`), same 75/25 split ratio, `random_state=42`
throughout so this is reproducible. **Before**: a naive `train_test_split` that ignores
`client_hash_id` entirely. **After**: the same `GroupShuffleSplit` the notebook already uses.

The gap itself is the finding here.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys
import pandas as pd
import numpy as np
import duckdb

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret before continuing."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN \'{HF_TOKEN}\')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT    = f"read_parquet(\'{WAREHOUSE}/dim_content.parquet\')"
FACT_APRIL_MAY = (
    f"read_parquet([\'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet\', "
    f"\'{WAREHOUSE}/fact_content_daily_performance/month=2026-05/*.parquet\'])"
)

# Same window as w05_model.ipynb: as-of 2026-05-31, decision moment 2026-05-01.
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE \'2026-05-31\' AS as_of_date
    ),
    per_item AS (
        SELECT f.client_hash_id, f.content_hash_id,
               MIN(f.report_date) AS first_seen,
               SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
               SUM(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
               AVG(CASE WHEN f.report_date <  b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_avg_position END)       AS prev_30_avg_position,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.gsc_clicks ELSE 0 END) AS prev_30_clicks,
               CASE
                   WHEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_impressions ELSE 0 END) > 0
                   THEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_clicks ELSE 0 END)
                        / SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                 THEN f.gsc_impressions ELSE 0 END)
               END AS prev_30_ctr,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.ga4_sessions ELSE 0 END) AS prev_30_sessions,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.ga4_engaged_sessions ELSE 0 END) AS prev_30_engaged_sessions,
               SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                        THEN f.scroll_events ELSE 0 END) AS prev_30_scroll_events
        FROM {FACT_APRIL_MAY} f, bounds b
        GROUP BY 1, 2
    )
    SELECT p.*, d.content_created_date, d.content_updated_date, d.keyword_created_date, d.keyword_char_count,
           d.keyword_token_count, d.content_type, d.search_volume, d.competition, d.cpc, d.main_intent,
           d.backlinks, d.category_count, d.char_count, d.word_count
    FROM per_item p
    JOIN {DIM_CONTENT} d USING (content_hash_id)
    WHERE p.first_seen <= DATE \'2026-05-31\' - INTERVAL 60 DAY   -- guard (a): full prev_30 history
      AND p.prev_30_impressions >= 100                             -- guard (b): activity floor
""").df()

decision_moment = pd.Timestamp("2026-05-01")
features["content_created_date"] = pd.to_datetime(features["content_created_date"])
features["content_updated_date"] = pd.to_datetime(features["content_updated_date"])
features["content_age_days_at_decision"] = (decision_moment - features["content_created_date"]).dt.days
features["days_since_last_update_at_decision"] = (decision_moment - features["content_updated_date"]).dt.days
features["keyword_created_date"] = pd.to_datetime(features["keyword_created_date"])
features["keyword_age_days_at_decision"] = (decision_moment - features["keyword_created_date"]).dt.days
features = features[features["days_since_last_update_at_decision"] > 0]   # guard (c)

features["impressions_pct_change"] = (
    (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
)
features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)

# EXACT feature set from w03_feature_leakage_check.ipynb section 2 / w05_model.ipynb
selected_features = [
    "prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
    "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
    "content_age_days_at_decision", "days_since_last_update_at_decision",
    "keyword_char_count", "keyword_token_count", "content_type",
    "search_volume", "competition", "cpc", "keyword_age_days_at_decision",
    "main_intent", "backlinks", "category_count", "char_count", "word_count",
]
df = features[selected_features].copy()

bins = [-np.inf, 1000, 2000, 3500, np.inf]
labels = ["<1000", "1000-2000", "2000-3500", "3500+"]
df["word_count_tier"] = pd.cut(df["word_count"], bins=bins, labels=labels, right=False).astype("object").fillna("NA")
bins = [-np.inf, 8000, 15000, 25000, np.inf]
labels = ["<8000", "8000-15000", "15000-25000", "25000+"]
df["char_count_tier"] = pd.cut(df["char_count"], bins=bins, labels=labels, right=False).astype("object").fillna("NA")
df = df.drop(columns=["word_count", "char_count"])

df["no_keyword_data"] = df["competition"].isna().astype(int)
df["search_volume"] = df["search_volume"].fillna(0)
df["competition"] = df["competition"].fillna(0)
df["cpc"] = df["cpc"].fillna(0)
df["keyword_age_days_at_decision"] = df["keyword_age_days_at_decision"].fillna(-30)
df["main_intent"] = df["main_intent"].fillna("NA")
df["backlinks_na"] = df["backlinks"].isna().astype(int)
df["backlinks"] = df["backlinks"].fillna(0)

y = features.loc[df.index, "is_declining"]
groups = features.loc[df.index, "client_hash_id"]

numeric_cols = ["prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
                "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
                "content_age_days_at_decision", "days_since_last_update_at_decision",
                "keyword_char_count", "keyword_token_count", "search_volume", "competition", "cpc",
                "keyword_age_days_at_decision", "backlinks", "category_count",
                "no_keyword_data", "backlinks_na"]
categorical_cols = ["content_type", "main_intent", "word_count_tier", "char_count_tier"]
assert set(numeric_cols + categorical_cols) == set(df.columns)

print(f"Feature frame: {len(df):,} rows, {df.shape[1]} columns, {groups.nunique()} clients.")
print(f"is_declining rate: {y.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 18,918 rows, 23 columns, 27 clients.
is_declining rate: 0.543


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order][:k].mean()

def make_pipes():
    lr = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), numeric_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)])),
                   ("clf", LogisticRegression(max_iter=2000, random_state=42))])
    tree = Pipeline([("prep", ColumnTransformer([("num", "passthrough", numeric_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)])),
                      ("clf", DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42))])
    return lr, tree

# ---- BEFORE: naive split, no grouping ----
Xtr_n, Xte_n, ytr_n, yte_n, gtr_n, gte_n = train_test_split(df, y, groups, test_size=0.25, random_state=42, stratify=y)
overlap_naive = len(set(gtr_n) & set(gte_n))
lr_n, tree_n = make_pipes()
lr_n.fit(Xtr_n, ytr_n); tree_n.fit(Xtr_n, ytr_n)
lr_p_n = lr_n.predict_proba(Xte_n)[:, 1]; tree_p_n = tree_n.predict_proba(Xte_n)[:, 1]

print("=== BEFORE: naive (random) split ===")
print(f"Client overlap between train/test: {overlap_naive} of {groups.nunique()} total clients")
print(f"Test base rate: {yte_n.mean():.3f}")
for k in [10, 50, 100]:
    print(f"  k={k:<4} LR precision@k={precision_at_k(lr_p_n, yte_n.values, k):.3f}   tree precision@k={precision_at_k(tree_p_n, yte_n.values, k):.3f}")
print(f"  LR AUC={roc_auc_score(yte_n, lr_p_n):.3f}   tree AUC={roc_auc_score(yte_n, tree_p_n):.3f}")

# ---- AFTER: honest client-grouped split (same as w05_model.ipynb) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups))
X_train, X_test = df.iloc[train_idx], df.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
lr_pipe, tree_pipe = make_pipes()
lr_pipe.fit(X_train, y_train); tree_pipe.fit(X_train, y_train)
lr_p_g = lr_pipe.predict_proba(X_test)[:, 1]; tree_p_g = tree_pipe.predict_proba(X_test)[:, 1]

print("\n=== AFTER: honest (client-grouped) split ===")
print(f"Client overlap between train/test: {overlap_grouped}")
print(f"Test base rate: {y_test.mean():.3f}")
for k in [10, 50, 100]:
    print(f"  k={k:<4} LR precision@k={precision_at_k(lr_p_g, y_test.values, k):.3f}   tree precision@k={precision_at_k(tree_p_g, y_test.values, k):.3f}")
print(f"  LR AUC={roc_auc_score(y_test, lr_p_g):.3f}   tree AUC={roc_auc_score(y_test, tree_p_g):.3f}")

print("\nExplaining the gap: the naive split let 24 of 27 clients appear in BOTH train and test --")
print("the model could partly memorize client-specific baselines (a client that's just generally")
print("high- or low-traffic) rather than learning signal that generalizes to a client it hasn't")
print("seen. That inflated AUC and precision@K across the board in the 'before' numbers above.")

=== BEFORE: naive (random) split ===
Client overlap between train/test: 24 of 27 total clients
Test base rate: 0.543
  k=10   LR precision@k=0.900   tree precision@k=0.800
  k=50   LR precision@k=0.820   tree precision@k=0.720
  k=100  LR precision@k=0.790   tree precision@k=0.640
  LR AUC=0.668   tree AUC=0.633

=== AFTER: honest (client-grouped) split ===
Client overlap between train/test: 0
Test base rate: 0.473
  k=10   LR precision@k=0.200   tree precision@k=0.800
  k=50   LR precision@k=0.500   tree precision@k=0.620
  k=100  LR precision@k=0.510   tree precision@k=0.650
  LR AUC=0.645   tree AUC=0.595

Explaining the gap: the naive split let 24 of 27 clients appear in BOTH train and test --
the model could partly memorize client-specific baselines (a client that's just generally
high- or low-traffic) rather than learning signal that generalizes to a client it hasn't
seen. That inflated AUC and precision@K across the board in the 'before' numbers above.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.